# Group 7: NFL Ad Package Optimization (Plan A)

A CBS affiliate sells season-long NFL ad packages: one 30-second ad in each of 27 games.
1. Before the NFL schedule is known, the station picks how much audience to guarantee,
   which prospects to pitch, and at what discount.
2. After the schedule is known, each salesperson splits limited hours between fixing
   ratings shortfalls and selling the remaining packages.

Goal: make as much money as possible while owing as few makegoods as possible.
These goals conflict, and this notebook shows the trade-off.

## Plan A at a glance
- **Role:** Main plan. It matches Ed's revised plan, where the station makes one guarantee
  decision for every package.
- **What's different from Plan B:** Plan B runs the model once per guarantee level and compares.
  Plan A lets the model choose the guarantee level itself.
- **Linear?** Yes. The model never multiplies two choices together (Step 4 explains how).
- **Course module:** Module 3 (two stages, scenarios, two objectives) plus Module 4
  (yes/no choices linked by "only if" rules). No big-M tricks are needed.
- **Size:** 345 decision variables and 167 constraints across 6 families. The rubric needs 50, 50, and 3.

## How this lines up with Ed's revised plan
| Ed's plan | This notebook | Note |
|---|---|---|
| i (advertiser) | prospects, i | Same |
| j (package/price option) | discounts, d | |
| s (Nielsen scenario) | scenarios, s | Same |
| r (salesperson) | salespeople, k | |
| x_i (sell to advertiser i) | sum over d of pitch[i,d,g] | We call it a pitch: the client buys with probability p_close, so it counts as a partial expected sale |
| y_i,j (price option j for advertiser i) | pitch[i,d,g] | |
| One station-wide guarantee decision | guarantee[g], a yes/no choice inside the model | |
| R_r (remediation) | fix | |
| S_r (additional selling) | sell | |
| h_r (hours) | hours[k] | Ed's doc says 8 hours; the spreadsheet says 12 |
| Maximize expected revenue minus UD cost | Two separate objectives | The rubric needs two conflicting objectives. Ed's expected profit is one point on our trade-off curve |

In [1]:
# SETUP
#   Import pandas (data), gurobipy (model), and matplotlib (chart).
#   Set file locations once, as relative paths, so every cell uses the same ones:
#     RAW_FILE  = data/raw/NFL_Ad_PKG_2.xlsx
#     CLEAN_DIR = data/clean/

## Step 1: Data
Data is the station's information, exactly as Ed provided it. We never edit the original file.
Code pulls out what we need, checks it against the spreadsheet's own totals, and saves
clean copies, so anyone can trace every number.

All audiences stay in thousands of households ("000s"), matching the spreadsheet.
Headers on these sheets are split across two rows or repeated, so we read columns by
letter and give them our own names.

In [2]:
# LOAD: 'Pre-Schedule Release', rows 13-17 (one row per game type)
#   What the station knew in July, before the schedule came out.
#   Skip the first 12 rows and read 5 rows. Row 18 is the totals row, which we skip.
#   col C = game type
#   col E = number of games
#   col F = rate per spot ($)
#   col H = audience per game (000s)
#
#   We need two numbers:
#     package_price = sum of (games x rate)       check: should equal 108000
#     baseline_000s = sum of (games x audience)   check: should equal 1631.3
#   Also check: games add up to 27.

In [3]:
# LOAD: 'Schedule Release 1-Min Qualifer', rows 13-39 (one row per actual game)
#   The real schedule, released in August. Low-interest games are already cut 30-40%.
#   Skip the first 12 rows and read 27 rows. Row 40 is the totals row, which we skip.
#   col C = game ID
#   col D = game name
#   col F = rate per spot ($)
#   col H = original audience per game, before cuts (000s)
#           Only needed if underdelivery is counted game by game (PENDING Ed).
#   col I = revised audience after low-interest cuts (000s)
#
#   We need:
#     revised_000s = sum of col I                 check: should equal 1465.938
#   Also check: 27 games, rates add up to 108000, and col H adds up to 1651.38.
#   (Col H isn't 1631.3 because the real schedule's game mix differs from July's.)
#
#   We skip cols M-O (the spreadsheet's bumped audiences). They're Excel formulas,
#   so we apply the increases ourselves in the scenario cell to show the steps.

In [4]:
# LOAD: 'Schedule Release 1-Min Qualifer', cells T6:U8 (one row per scenario)
#   Nielsen's new 1-minute rule raises ratings, but nobody knows by how much.
#   col T = label, such as "15% Increase" -> convert to a number (0.15)
#   col U = probability of that increase
#
#   We need, for each scenario:
#     increase[s], prob[s]
#     stage1_delivered_000s[s] = baseline_000s x (1 + increase[s])
#       What July expects: generic games, no dud cuts yet (Ed's revised plan).
#       check: 1875.995 / 1957.56 / 2039.125
#     stage2_delivered_000s[s] = revised_000s x (1 + increase[s])
#       What August expects: the real schedule, dud games included.
#       check: 1685.8287 / 1759.1256 / 1832.4225
#   Also check: probabilities add up to 1.

In [5]:
# LOAD: 'Prospect Data Table', rows 4-33 (one row per prospect)
#   The 30 potential clients.
#   Skip the first 3 rows and read 30 rows.
#   col B = prospect ID
#   col C = prospect name
#   col D = salesperson who owns them
#   col E = base chance of buying
#   col F = goal CPM ($ they want to pay per 1000 viewers)
#   col G = makegood penalty multiplier (2, 3, or 4)
#
#   Check: 30 prospects, 6 salespeople, 5 prospects each.
#   NOTE: Use this table's 2x/3x/4x penalties. Matt's prototype uses an older
#         table with 0.05-0.25 penalties.

In [6]:
# LOAD: 'Salesperson Table', rows 5-9 (one row per salesperson)
#   Time available in the key week after the schedule comes out.
#   Skip the first 4 rows and read 5 rows.
#   col B = salesperson
#   col C = close ratio (should match the Prospect table; use that table's copy)
#   col D = hours available
#   col F = hours per makegood fix
#   col G = hours per new sale
#
#   Check: close ratios match the Prospect table (they do for SP1-SP5).
#   FLAG: SP6 is missing, but owns 5 prospects. Waiting on Ed.
#   FLAG: cols F and G have the same label. We're assuming F = fix, G = sale.
#   FLAG: Ed's revised plan uses 3.5 hrs per fix, 3 per sale, and 8 available.
#         The spreadsheet says 3.75, 5, and 12. We use the spreadsheet until decided.

In [7]:
# SAVE CLEAN FILES to CLEAN_DIR
#   package.csv      package_price, baseline_000s, revised_000s
#   games.csv        27 rows from the schedule cell (original and revised audience)
#   scenarios.csv    3 rows: increase, prob, stage1_delivered_000s, stage2_delivered_000s
#   prospects.csv    30 rows from the prospect cell
#   salespeople.csv  6 rows from the salesperson cell (once SP6 is added)
# Everything from here on reads only these files.

In [8]:
# TEAM ASSUMPTIONS: numbers the spreadsheet doesn't contain
#   Kept in one place so they're easy to find, change, and cite in the report.
#   discounts         [0, 0.05, 0.10]            team decision
#   guarantee_levels  [0, 0.15, 0.25]            plan doc; PENDING Ed
#                     In Plan A, these are the options the model chooses from.
#                     NOTE: in the July view, a guarantee only falls short if it's above
#                     the smallest bump (15%). Of these levels, only 0.25 does.
#   max_presales      10                         team decision
#   total_packages    20                         20 ad slots per game, per Ed
#   penalty_slope     0.0025 per 1% over goal    plan doc; CONFIRM with Ed
#   local_news_rate   1.75                       Ed: a fix swaps the full makegood for Local
#                                                News at 1.75x the shortfall. Ed dropped the
#                                                75% acceptance rate, so a fix always works.
#   stage2_price      PENDING Ed (assume 108000)
#
#   DECIDE on Sunday:
#     - Does max_presales limit pitches or expected sales?
#     - Goal CPM: a hard cutoff (no pitch if over goal, like Matt's prototype),
#       or a lower chance of closing (penalty_slope)?
#     - Is Stage 2 planned per Nielsen scenario (as written), or on expected delivery?
#     - Are fixes tracked per salesperson (as written), or per client?

## Step 2: Sets

Sets are the lists of things the model makes decisions about.
Each set works like a roster, and the model loops over it.

In [9]:
# SETS
#   prospects         <- ID column of prospects.csv              check: 30
#   discounts         <- team assumptions cell                   check: 3
#   guarantee_levels  <- team assumptions cell                   check: 3
#   salespeople       <- salesperson column of salespeople.csv   check: 6
#   scenarios         <- rows of scenarios.csv                   check: 3
#   owned_by[k]       <- prospects.csv grouped by salesperson    check: 5 each
#
#   Unlike Plan B, guarantee_levels IS a set here, because the model chooses from it.

## Step 3: Parameters
Parameters are fixed numbers the model uses but cannot change: prices, hours, probabilities.
Some come straight from the clean files, and others are calculated from them.

In [10]:
# PARAMETERS FROM THE CLEAN FILES
#   Store each as a lookup (dictionary) keyed by its set, so the model can use it.
#   package_price, baseline_000s                        package.csv
#   stage1_delivered_000s[s], stage2_delivered_000s[s],
#   prob[s]                                             scenarios.csv
#   close[i], goal_cpm[i], penalty[i]                   prospects.csv
#   hours[k], hours_fix, hours_sell                     salespeople.csv
#   max_presales, total_packages, stage2_price,
#   penalty_slope, local_news_rate                      team assumptions cell

In [11]:
# CALCULATED PARAMETERS, computed for EVERY guarantee level g at once
#   Plan B recalculated these inside a loop. Plan A needs all of them up front,
#   because the model compares every guarantee level in a single solve.
#
#   guaranteed_000s[g]    = baseline_000s x (1 + g)
#   offered_cpm[d,g]      = package_price x (1 - d) / guaranteed_000s[g]
#                           check: d = 0, g = 0 gives about 66.20
#                                  d = 0.10, g = 0.25 gives about 47.67
#   cpm_gap[i,d,g]        = (offered_cpm[d,g] - goal_cpm[i]) / goal_cpm[i] x 100
#                           (set to 0 if at or below goal)
#   p_close[i,d,g]        = close[i] - penalty_slope x cpm_gap[i,d,g]
#                           (set to 0 if negative) check: never above close[i]
#                           (If we choose a hard cutoff instead: p_close = close[i] when at
#                           or below goal, and 0 when over.)
#   exp_rev[i,d,g]        = p_close[i,d,g] x package_price x (1 - d)
#
#   shortfall_stage1[g,s] = guaranteed_000s[g] - stage1_delivered_000s[s]   (0 if negative)
#                           July view: generic games only.
#                           check: g = 0 and g = 0.15 give 0 / 0 / 0
#                                  g = 0.25 gives about 163.13 / 81.57 / 0
#   shortfall_stage2[g,s] = August view: real schedule, dud games included.
#                           On the package total:
#                             guaranteed_000s[g] - stage2_delivered_000s[s]   (0 if negative)
#                             check: g = 0    gives 0 / 0 / 0
#                                    g = 0.15 gives about 190.17 / 116.87 / 43.57
#                                    g = 0.25 gives about 353.30 / 280.00 / 206.70
#                           Game by game (PENDING Ed): for each game,
#                             original x (1 + g) - revised x (1 + increase[s]),
#                             keep only shortfalls, then add up the games.
#                             check: g = 0    gives about 122.49 / 101.51 / 80.53
#                                    g = 0.15 gives about 213.26 / 192.28 / 171.29
#                                    g = 0.25 gives about 378.40 / 305.10 / 231.80
#
#   mg_rate[g]              = package_price / guaranteed_000s[g]
#                             $ per thousand owed impressions (package CPM; CONFIRM with Ed)
#   exp_mg_stage1[i,d,g,s]  = p_close[i,d,g] x penalty[i] x shortfall_stage1[g,s] x mg_rate[g]
#   exp_mg_stage2[i,d,g,s]  = p_close[i,d,g] x penalty[i] x shortfall_stage2[g,s] x mg_rate[g]
#   fix_savings[k,g,s]      = (average penalty of k's prospects - local_news_rate)
#                             x shortfall_stage2[g,s] x mg_rate[g]
#                             What one fix saves: the full makegood minus Local News at 1.75x.
#                             (If fixes are tracked per client, use penalty[i] instead.)

## Decision for Sunday: one model or two?
Ed's plan says the station can't see the schedule in July. So Stage 1 should judge risk with the
July view (`shortfall_stage1`), and Stage 2 should react to the real schedule (`shortfall_stage2`).

- **One model (as written below):** Both stages are solved together, and makegood cost uses
  `exp_mg_stage2`. It's simpler and bigger for the rubric, but Stage 1 effectively "knows" the dud
  games in July. We'd state that as a limitation in the report.
- **Two models in sequence:** Model 1 picks pitches using `exp_mg_stage1`. Those pitches are then
  locked in, and Model 2 decides sell and fix using `exp_mg_stage2`. This matches Ed's story, but
  the trade-off and rubric counts have to work across both models.

## Step 4: Decision variables
Decision variables are the choices Gurobi makes.
Stage 1 choices happen before ratings are known: one plan for every scenario.
Stage 2 choices happen after: a separate plan for each scenario.

**How Plan A stays linear.** The chance a prospect buys depends on both the discount *and*
the guarantee. If "which discount" and "which guarantee" were two separate choices, the model
would have to multiply them together to get the chance of a sale. A linear model can't multiply
two choices. Instead, every discount-and-guarantee combination gets its own pitch choice, with its
own fixed chance of closing from Step 3. A rule in Step 5 then allows pitches only under the
guarantee the station actually picks.

In [12]:
# BUILD THE MODEL AND ITS VARIABLES
#   Unlike Plan B, the model is built once. Nothing needs rebuilding per guarantee level.
#
#   Stage 1 (before the schedule)
#   guarantee[g]    yes/no: the station picks guarantee level g
#                   count = guarantee_levels = 3
#   pitch[i,d,g]    yes/no: pitch prospect i at discount d under guarantee g
#                   count = prospects x discounts x guarantee_levels = 270
#
#   Stage 2 (after the schedule, one plan per scenario)
#   sell[k,s]       whole number: new packages salesperson k sells in scenario s
#                   count = salespeople x scenarios = 18
#   fix[k,g,s]      whole number: makegood fixes salesperson k does in scenario s,
#                   on clients sold under guarantee g
#                   count = salespeople x guarantee_levels x scenarios = 54
#                   Fixes carry g because what a fix saves depends on the shortfall,
#                   and the shortfall depends on the guarantee.
#
#   Total: 345
#   A pitch isn't a sure sale. It counts as p_close[i,d,g] of an expected sale.
#
#   If fixes are tracked per client instead:
#   fix[i,g,s]      count = prospects x guarantee_levels x scenarios = 270,
#                   so the total becomes 561.

## Step 5: Constraints
Constraints are the rules every solution must follow. A family is one rule applied
across many prospects, salespeople, or scenarios.

Two rules connect everything to the guarantee choice without multiplying variables:
- Rule 2 allows pitches only under the chosen guarantee.
- Rule 6 allows fixes only on clients who were actually pitched under that guarantee.

When a guarantee level isn't chosen, both rules force its pitches and fixes to zero.

In [13]:
# CONSTRAINTS
#   1. Pick exactly one guarantee level
#      sum over g of guarantee[g] = 1
#      count = 1
#
#   2. Pitch only under the chosen guarantee, at one price at most
#      For each prospect i and level g: sum over d of pitch[i,d,g] <= guarantee[g]
#      If g is chosen, the right side is 1, so at most one discount.
#      If g isn't chosen, the right side is 0, so no pitches under g.
#      This also does Plan B's "one price per prospect" job.
#      count = prospects x guarantee_levels = 90
#
#   3. Presale limit (DECIDE which version)
#      sum of pitch <= max_presales, OR sum of p_close x pitch <= max_presales
#      count = 1
#
#   4. Salesperson hours
#      For each k and s:
#      hours_fix x (sum over g of fix[k,g,s]) + hours_sell x sell[k,s] <= hours[k]
#      count = salespeople x scenarios = 18
#
#   5. Inventory
#      For each s: sum of p_close x pitch (over all i, d, g) + sum over k of sell[k,s]
#                  <= total_packages
#      count = scenarios = 3
#
#   6. Can't fix clients you don't have
#      For each k, g, and s:
#      fix[k,g,s] <= sum over owned_by[k] and d of p_close[i,d,g] x pitch[i,d,g]
#      If g isn't chosen, every pitch under g is 0, so fixes under g are 0 too.
#      count = salespeople x guarantee_levels x scenarios = 54
#
#   Total: 167 constraints across 6 families.
#
#   If fixes are tracked per client: rule 6 becomes, for each i, g, and s,
#   fix[i,g,s] <= sum over d of p_close[i,d,g] x pitch[i,d,g] (count 270), and rule 4
#   adds up fixes over owned_by[k]. The total becomes 383.

## Step 6: Objectives
Objectives are what we're trying to achieve. Ours conflict.

In [14]:
# OBJECTIVE 1, maximize expected revenue:
#   sum of exp_rev[i,d,g] x pitch[i,d,g]
#   + sum over s of prob[s] x stage2_price x sum over k of sell[k,s]
#
# OBJECTIVE 2, minimize expected makegood cost:
#   sum over s of prob[s] x (sum of exp_mg_stage2[i,d,g,s] x pitch[i,d,g]
#                            - sum over k and g of fix_savings[k,g,s] x fix[k,g,s])
#   (One-model version. In the two-model version, Model 1 uses exp_mg_stage1.)
#
# Conflict:
#   - A bigger guarantee makes the package look cheaper to clients, so more buy,
#     but it also raises the chance the station falls short.
#   - Presales add revenue but also makegood risk.
#   - Fixes cut cost but use hours that could have sold packages.
#
# Ed's expected profit (revenue minus makegood cost) isn't a third objective.
# It's one point on our trade-off curve, where both objectives count equally.

In [15]:
# RUBRIC CHECK
#   Build the model and print Gurobi's own counts.
#   check: variables = 345, constraints = 167, both at least 50

## Step 7: Trade-off
We can't get the best of both objectives at once, so we map the best possible
combinations and choose one. In Plan A, the model also picks the guarantee level at
each point, so the chart shows where the best guarantee switches from safe to aggressive.

In [16]:
# TRADE-OFF LOOP (no outer loop over guarantee levels, since the model chooses)
#   1. Maximize revenue alone. Record revenue, makegood cost, and the guarantee chosen.
#   2. Minimize makegood cost alone. Record the same three things.
#      Expect this run to pitch almost no one: no presales means no makegoods.
#      That's the zero-risk end of the curve.
#   3. Add one extra rule: makegood cost <= budget.
#   4. Pick about 10 budgets between the two costs from steps 1 and 2. For each one,
#      change the budget rule's limit and maximize revenue again.
#   5. Save each run: budget, revenue, makegood cost, guarantee chosen,
#      who was pitched at what discount, and sell/fix by salesperson and scenario.

In [17]:
# RESULTS
#   Combine all saved runs into one table, and save it to CLEAN_DIR as results.csv.
#   Chart revenue (vertical) against makegood cost (horizontal), as one line.
#   Label each point with the guarantee level the model chose.
#   Mark the point with the highest revenue minus makegood cost (Ed's expected profit).
#   Our pick is the point where more makegood risk stops adding much revenue.
#   The report should explain which guarantee level that point uses, and why.

## Open questions and decisions

**Answered by Ed**
- Stage 1 still selects which clients to sell to, and at what price.
- The station makes one guarantee decision for every package.
- Drop the 75% acceptance rate: a Local News fix always works.

**Still open for Ed**
1. Is underdelivery counted game by game, or on the package total? This decides most of the Stage 2 numbers.
2. Which guarantee levels would the station actually consider? With 0/15/25%, only 25% ever falls short in the July view.
3. What dollar rate turns owed impressions into cost? We're proposing the package CPM (about 66 dollars per thousand).
4. Hours: his revised plan says 3.5 per fix, 3 per sale, and 8 available. The spreadsheet says 3.75, 5, and 12. Which is right?
5. What are SP6's hours, and is $108K the right Stage 2 package price?

**For the group to decide on Sunday**
1. One model, or two in sequence? (See the cell after Step 3.)
2. Presale limit: cap pitches, or expected sales?
3. Goal CPM: hard cutoff, or a lower chance of closing?
4. Stage 2: planned per Nielsen scenario, or on expected delivery?
5. Fixes: tracked per salesperson, or per client?
6. Plan A or Plan B? This notebook matches Ed's plan; Plan B is the backup.